# Compare Processed Features vs `allcel`

This notebook compares features computed in processed data against the original per-cell values stored in the dataset.
If the implementation is consistent, points should lie close to the diagonal `y = x`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True

processed_root = Path('../data/processed')
feature_paths = sorted(processed_root.rglob('*_features.parquet'))
print(f'Found {len(feature_paths)} feature parquet files under {processed_root.resolve()}')

In [ ]:
if not feature_paths:
    raise FileNotFoundError('No feature parquets found. Run scripts/interim_to_processed.py first.')

df = pd.concat((pd.read_parquet(p) for p in feature_paths), ignore_index=True)
print(f'Total rows: {len(df):,}')
print(f'Total columns: {df.shape[1]}')
display(df.head(3))

In [ ]:
comparisons = [
    # (our_column, allcel_column, panel_title)
    ('burst_index', 'allcel__burst_u', 'Burst index'),
    ('fr_hz', 'allcel__fr_u', 'Firing rate (Hz)'),
    ('spk_duration_ms', 'allcel__duration_u', 'Spike duration (ms)'),
    ('spk_asymmetry', 'allcel__assymetry_u', 'Spike asymmetry'),
]

missing = [c for c in comparisons if c[0] not in df.columns or c[1] not in df.columns]
if missing:
    print('Missing columns for some comparisons:')
    for ours, base, title in missing:
        print(f'  - {title}: ours={ours in df.columns}, allcel={base in df.columns}')

valid_comparisons = [c for c in comparisons if c not in missing]
if not valid_comparisons:
    raise ValueError('None of the requested comparison columns are present.')

n = len(valid_comparisons)
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes = axes.ravel()

summary_rows = []
for i, (ours_col, base_col, title) in enumerate(valid_comparisons):
    ax = axes[i]
    xy = df[[ours_col, base_col]].apply(pd.to_numeric, errors='coerce').dropna()

    if xy.empty:
        ax.text(0.5, 0.5, 'No valid numeric rows', ha='center', va='center')
        ax.set_axis_off()
        continue

    x = xy[base_col].to_numpy()
    y = xy[ours_col].to_numpy()

    ax.scatter(x, y, s=8, alpha=0.25, edgecolors='none')

    lo = float(np.nanmin([x.min(), y.min()]))
    hi = float(np.nanmax([x.max(), y.max()]))
    if np.isfinite(lo) and np.isfinite(hi):
        pad = 0.03 * (hi - lo if hi > lo else 1.0)
        lo_p, hi_p = lo - pad, hi + pad
        ax.plot([lo_p, hi_p], [lo_p, hi_p], 'r--', lw=1.2, label='y = x')
        ax.set_xlim(lo_p, hi_p)
        ax.set_ylim(lo_p, hi_p)

    if len(xy) >= 2:
        r = float(np.corrcoef(x, y)[0, 1])
        slope, intercept = np.polyfit(x, y, 1)
    else:
        r = np.nan
        slope, intercept = np.nan, np.nan

    ax.set_title(title)
    ax.set_xlabel(f'{base_col} (dataset)')
    ax.set_ylabel(f'{ours_col} (processed)')
    ax.legend(frameon=False, loc='best', fontsize=8)
    ax.text(
        0.03, 0.97,
        f'n={len(xy):,}\nr={r:.4f}\nslope={slope:.4f}\nintercept={intercept:.4f}',
        transform=ax.transAxes,
        va='top',
        ha='left',
        fontsize=8,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8),
    )

    summary_rows.append({
        'feature': title,
        'ours_col': ours_col,
        'allcel_col': base_col,
        'n': len(xy),
        'pearson_r': r,
        'fit_slope': float(slope),
        'fit_intercept': float(intercept),
    })

for j in range(n, len(axes)):
    axes[j].axis('off')

fig.suptitle('Processed features vs dataset (`allcel`) values', fontsize=12)
fig.tight_layout()
plt.show()

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
# Histogram of processed spike asymmetry
asym = pd.to_numeric(df['spk_asymmetry'], errors='coerce').dropna()

if asym.empty:
    print('No valid spk_asymmetry values found.')
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(asym, bins=80, color='tab:blue', alpha=0.85, edgecolor='white')
    ax.set_xlabel('spk_asymmetry')
    ax.set_ylabel('count')
    ax.set_title('Distribution of processed spk_asymmetry')
    ax.text(0.98, 0.95, f'n={len(asym):,}', transform=ax.transAxes, ha='right', va='top')
    fig.tight_layout()
    plt.show()
